In [ ]:
# !pip install -q torch==2.5.1 torchvision torchaudio ultralyticsplus==0.0.28 ultralytics==8.0.43
# !pip install -q "numpy<2.0.0" supervision motmetrics

In [84]:

import os
import cv2
import yaml
import shutil
import numpy as np
import pandas as pd
import motmetrics as mm
import supervision as sv
from pathlib import Path
from ultralytics import YOLO
from ultralyticsplus import YOLO, render_result

import torch
import torchvision.transforms as T
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.ops import roi_align

from supervision.detection.core import Detections
from supervision.detection.utils.iou_and_nms import box_iou_batch
from supervision.tracker.byte_tracker.single_object_track import STrack, TrackState
from supervision.tracker.byte_tracker import matching
from supervision.tracker.byte_tracker.core import joint_tracks, sub_tracks, remove_duplicate_tracks

## YOLO Dataset preparation

Data is from:
https://github.com/VisDrone/VisDrone-Dataset.git

In [2]:
base_path = "/home/oleh/studying/3.2/CV/cv-midterm/task1/VisDrone2019-MOT-test-dev"
sequences_path = os.path.join(base_path, "sequences")
annotations_path = os.path.join(base_path, "annotations")
dataset_dir = "/home/oleh/studying/3.2/CV/cv-midterm/task1/yolo_dataset"

In [3]:
def load_visdrone_annotations_lenient(ann_file):

    annotations_by_frame = {}
    ignored_count = 0

    with open(ann_file, 'r') as f:
        for line in f:
            parts = line.strip().split(',')
            
            frame_id = int(parts[0])
            target_id = int(parts[1])
            x = float(parts[2])
            y = float(parts[3])
            w = float(parts[4])
            h = float(parts[5])
    
            conf = float(parts[6])
            class_id = int(parts[7])
            truncation = int(parts[8])
            occlusion = int(parts[9])

            if class_id == 0 or class_id == 11:
                ignored_count += 1
                continue

            if w <= 0 or h <= 0 or x < 0 or y < 0:
                continue
            
            if frame_id not in annotations_by_frame:
                annotations_by_frame[frame_id] = []
            
            annotations_by_frame[frame_id].append({
                'x': x, 'y': y, 'w': w, 'h': h, 'conf': conf, 'class': class_id
            })
    
    return annotations_by_frame, ignored_count

def convert_to_yolo_format(x, y, w, h, img_width, img_height):

    center_x = (x + w/2) / img_width
    center_y = (y + h/2) / img_height
    norm_w = w / img_width
    norm_h = h / img_height

    center_x = max(0.001, min(0.999, center_x))
    center_y = max(0.001, min(0.999, center_y))
    norm_w = max(0.001, min(1.0, norm_w))
    norm_h = max(0.001, min(1.0, norm_h))
    
    return center_x, center_y, norm_w, norm_h


In [4]:
if os.path.exists(dataset_dir):
    shutil.rmtree(dataset_dir)
    print(f"Cleaned old dataset directory")

os.makedirs(os.path.join(dataset_dir, "images", "train"), exist_ok=True)
os.makedirs(os.path.join(dataset_dir, "images", "val"), exist_ok=True)
os.makedirs(os.path.join(dataset_dir, "labels", "train"), exist_ok=True)
os.makedirs(os.path.join(dataset_dir, "labels", "val"), exist_ok=True)

sequence_dirs = sorted([d for d in os.listdir(sequences_path) if os.path.isdir(os.path.join(sequences_path, d))])
train_count = 0
val_count = 0

print(f"Processing {len(sequence_dirs)} sequences...")

Processing 17 sequences...


In [5]:
for seq_idx, seq_dir in enumerate(sequence_dirs):
    seq_path = os.path.join(sequences_path, seq_dir)
    ann_file = os.path.join(annotations_path, f"{seq_dir}.txt")

    annotations_by_frame, ignored = load_visdrone_annotations_lenient(ann_file)

    image_files = sorted([f for f in os.listdir(seq_path) if f.endswith(('.jpg', '.png'))])

    is_train = (seq_idx % 5) != 4
    split_type = "train" if is_train else "val"

    seq_train = 0
    seq_val = 0

    for img_file in image_files:
        img_path = os.path.join(seq_path, img_file)

        try:
            frame_id = int(os.path.splitext(img_file)[0])
        except ValueError:
            continue

        if frame_id not in annotations_by_frame:
            continue

        img = cv2.imread(img_path)
        if img is None:
            continue

        h, w = img.shape[:2]

        yolo_labels = []
        for det in annotations_by_frame[frame_id]:
            cx, cy, norm_w, norm_h = convert_to_yolo_format(
                det['x'], det['y'], det['w'], det['h'], w, h
            )
            yolo_labels.append(f"{det['class'] - 1} {cx:.6f} {cy:.6f} {norm_w:.6f} {norm_h:.6f}")

        if not yolo_labels:
            continue

        img_name = f"{seq_dir}_{img_file}"
        dst_img = os.path.join(dataset_dir, "images", split_type, img_name)
        dst_label = os.path.join(dataset_dir, "labels", split_type, img_name.replace(os.path.splitext(img_file)[1], '.txt'))

        cv2.imwrite(dst_img, img)
        with open(dst_label, 'w') as f:
            f.write('\n'.join(yolo_labels))

        if is_train:
            train_count += 1
            seq_train += 1
        else:
            val_count += 1
            seq_val += 1

    total_seq = seq_train + seq_val
    if total_seq > 0:
        print(f"  ✓ {seq_dir}: {total_seq} images ({seq_train} train, {seq_val} val)")


  ✓ uav0000009_03358_v: 219 images (219 train, 0 val)
  ✓ uav0000073_00600_v: 328 images (328 train, 0 val)
  ✓ uav0000073_04464_v: 312 images (312 train, 0 val)
  ✓ uav0000077_00720_v: 780 images (780 train, 0 val)
  ✓ uav0000088_00290_v: 296 images (0 train, 296 val)
  ✓ uav0000119_02301_v: 179 images (179 train, 0 val)
  ✓ uav0000120_04775_v: 1000 images (1000 train, 0 val)
  ✓ uav0000161_00000_v: 308 images (308 train, 0 val)
  ✓ uav0000188_00000_v: 260 images (260 train, 0 val)
  ✓ uav0000201_00000_v: 677 images (0 train, 677 val)
  ✓ uav0000249_00001_v: 360 images (360 train, 0 val)
  ✓ uav0000249_02688_v: 244 images (244 train, 0 val)
  ✓ uav0000297_00000_v: 146 images (146 train, 0 val)
  ✓ uav0000297_02761_v: 373 images (373 train, 0 val)
  ✓ uav0000306_00230_v: 420 images (0 train, 420 val)
  ✓ uav0000355_00001_v: 468 images (468 train, 0 val)
  ✓ uav0000370_00001_v: 265 images (265 train, 0 val)


## a) YOLO detection

In [29]:

print(f"\n✅ Dataset ready: {train_count} train, {val_count} validation ({train_count+val_count} total)")

data_yaml = {
    'path': dataset_dir,
    'train': 'images/train',
    'val': 'images/val',
    'nc': 10,
    'names': ['pedestrian', 'people', 'bicycle', 
              'car', 'van', 'truck', 'tricycle', 
              'awning-tricycle', 'bus', 'motor']
}

yaml_path = os.path.join(dataset_dir, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"✓ YOLO config saved to {yaml_path}")
print(f"Dataset structure ready for training!")


✅ Dataset ready: 5242 train, 1393 validation (6635 total)
✓ YOLO config saved to /home/oleh/studying/3.2/CV/cv-midterm/task1/yolo_dataset/data.yaml
Dataset structure ready for training!


In [ ]:
yaml_path = os.path.join(dataset_dir, 'data.yaml')

model = YOLO('yolov8m.pt')

results = model.train(
    data=yaml_path,
    epochs=200,
    imgsz=1024,
    batch=8,
    device=0,               
    patience=30,
    save=True,              
    save_period=10,         
    project='yolo_training',  
    name='visdrone_run1',   
    exist_ok=False,         
    pretrained=True,             
    momentum=0.937,         
    weight_decay=0.0005,    
    warmup_epochs=3,        
    warmup_momentum=0.8,    
    warmup_bias_lr=0.1,     
    box=7.5,                
    cls=0.5,                
    dfl=1.5,                
    hsv_h=0.015,            
    hsv_s=0.7,              
    hsv_v=0.4,              
    degrees=10.0,           
    translate=0.1,          
    scale=0.5,              
    flipud=0.0,             
    fliplr=0.5,             
    mosaic=1.0,             
    mixup=0.0,              
    copy_paste=0.0,         
    verbose=True,           
    close_mosaic=10         
)

print("✓ Training completed!")
print(f"Results saved to 'yolo_training/visdrone_run1'")

Starting YOLO training...
New https://pypi.org/project/ultralytics/8.4.34 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.33 🚀 Python-3.12.3 torch-2.11.0+cu130 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 7808MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/oleh/studying/3.2/CV/cv-midterm/task1/yolo_dataset/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=tr

RuntimeError: CUDA error: CUBLAS_STATUS_NOT_SUPPORTED when calling `cublasGemmEx( handle, opa, opb, m, n, k, &falpha, a, CUDA_R_16BF, lda, b, CUDA_R_16BF, ldb, &fbeta, c, std::is_same_v<C_Dtype, float> ? CUDA_R_32F : CUDA_R_16BF, ldc, compute_type, CUBLAS_GEMM_DEFAULT_TENSOR_OP)`

## b) ByteTrack

In [6]:
model = YOLO('mshamrai/yolov8n-visdrone')

model.overrides['conf'] = 0.25
model.overrides['iou'] = 0.45
model.overrides['agnostic_nms'] = False
model.overrides['max_det'] = 1000

tracker = sv.ByteTrack(
    track_activation_threshold=0.25,
    lost_track_buffer=30,
    minimum_matching_threshold=0.8,
    frame_rate=30
)

box_annotator = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()
mot_file_path = "detection_metrics.txt"

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/mshamrai/yolov8n-visdrone/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/mshamrai/yolov8n-visdrone/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/mshamrai/yolov8n-visdrone/28ca5101e5e6090c6beba15eb79d038d0befad6d/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/mshamrai/yolov8n-visdrone/resolve/main/best.pt "HTTP/1.1 302 Found"


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
reid_model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
reid_model.fc = torch.nn.Identity()
reid_model.to(device).eval()

preprocess = T.Compose([
    T.ToPILImage(),
    T.Resize((256, 128)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:


def update_with_tensors(self, tensors: np.ndarray) -> list[STrack]:

    self.frame_id += 1
    activated_starcks = []
    refind_stracks = []
    lost_stracks = []
    removed_stracks = []

    scores = tensors[:, 4]
    bboxes = tensors[:, :4]

    remain_inds = scores > self.track_activation_threshold
    inds_low = scores > 0.1
    inds_high = scores < self.track_activation_threshold

    inds_second = np.logical_and(inds_low, inds_high)
    dets_second = bboxes[inds_second]
    dets = bboxes[remain_inds]
    scores_keep = scores[remain_inds]
    scores_second = scores[inds_second]

    if len(dets) > 0:
        """Detections"""
        detections = [
            STrack(
                STrack.tlbr_to_tlwh(tlbr),
                score_keep,
                self.minimum_consecutive_frames,
                self.shared_kalman,
                self.internal_id_counter,
                self.external_id_counter,
            )
            for (tlbr, score_keep) in zip(dets, scores_keep)
        ]
    else:
        detections = []

    """ Add newly detected tracklets to tracked_stracks"""
    unconfirmed = []
    tracked_stracks = []  # type: list[STrack]

    for track in self.tracked_tracks:
        if not track.is_activated:
            unconfirmed.append(track)
        else:
            tracked_stracks.append(track)

    """ Step 2: First association, with high score detection boxes"""
    strack_pool = joint_tracks(tracked_stracks, self.lost_tracks)
    # Predict the current location with KF
    STrack.multi_predict(strack_pool, self.shared_kalman)
    dists = matching.iou_distance(strack_pool, detections)

    dists = matching.fuse_score(dists, detections)
    matches, u_track, u_detection = matching.linear_assignment(
        dists, thresh=self.minimum_matching_threshold
    )

    for itracked, idet in matches:
        track = strack_pool[itracked]
        det = detections[idet]
        if track.state == TrackState.Tracked:
            track.update(detections[idet], self.frame_id)
            activated_starcks.append(track)
        else:
            track.re_activate(det, self.frame_id)
            refind_stracks.append(track)

    """ Step 3: Second association, with low score detection boxes"""
    # association the untrack to the low score detections
    if len(dets_second) > 0:
        """Detections"""
        detections_second = [
            STrack(
                STrack.tlbr_to_tlwh(tlbr),
                score_second,
                self.minimum_consecutive_frames,
                self.shared_kalman,
                self.internal_id_counter,
                self.external_id_counter,
            )
            for (tlbr, score_second) in zip(dets_second, scores_second)
        ]
    else:
        detections_second = []
    r_tracked_stracks = [
        strack_pool[i]
        for i in u_track
        if strack_pool[i].state == TrackState.Tracked
    ]
    dists = matching.iou_distance(r_tracked_stracks, detections_second)
    matches, u_track, u_detection_second = matching.linear_assignment(
        dists, thresh=0.5
    )
    for itracked, idet in matches:
        track = r_tracked_stracks[itracked]
        det = detections_second[idet]
        if track.state == TrackState.Tracked:
            track.update(det, self.frame_id)
            activated_starcks.append(track)
        else:
            track.re_activate(det, self.frame_id)
            refind_stracks.append(track)

    for it in u_track:
        track = r_tracked_stracks[it]
        if not track.state == TrackState.Lost:
            track.state = TrackState.Lost
            lost_stracks.append(track)

    """Deal with unconfirmed tracks, usually tracks with only one beginning frame"""
    detections = [detections[i] for i in u_detection]
    dists = matching.iou_distance(unconfirmed, detections)

    dists = matching.fuse_score(dists, detections)
    matches, u_unconfirmed, u_detection = matching.linear_assignment(
        dists, thresh=0.7
    )
    for itracked, idet in matches:
        unconfirmed[itracked].update(detections[idet], self.frame_id)
        activated_starcks.append(unconfirmed[itracked])
    for it in u_unconfirmed:
        track = unconfirmed[it]
        track.state = TrackState.Removed
        removed_stracks.append(track)

    """ Step 4: Init new stracks"""
    for inew in u_detection:
        track = detections[inew]
        if track.score < self.det_thresh:
            continue
        track.activate(self.kalman_filter, self.frame_id)
        activated_starcks.append(track)
    """ Step 5: Update state"""
    for track in self.lost_tracks:
        if self.frame_id - track.frame_id > self.max_time_lost:
            track.state = TrackState.Removed
            removed_stracks.append(track)

    self.tracked_tracks = [
        t for t in self.tracked_tracks if t.state == TrackState.Tracked
    ]
    self.tracked_tracks = joint_tracks(self.tracked_tracks, activated_starcks)
    self.tracked_tracks = joint_tracks(self.tracked_tracks, refind_stracks)
    self.lost_tracks = sub_tracks(self.lost_tracks, self.tracked_tracks)
    self.lost_tracks.extend(lost_stracks)
    self.lost_tracks = sub_tracks(self.lost_tracks, self.removed_tracks)
    self.removed_tracks = removed_stracks
    self.tracked_tracks, self.lost_tracks = remove_duplicate_tracks(
        self.tracked_tracks, self.lost_tracks
    )
    output_stracks = [track for track in self.tracked_tracks if track.is_activated]

    return output_stracks

In [ ]:
def extract_embeddings(frame, boxes):
    """
    frame: cv2 image
    boxes: numpy array (N, 4) в форматі xyxy
    """
    if len(boxes) == 0:
        return torch.empty((0, 2048)).to(device)

    rois = torch.zeros((len(boxes), 5)).to(device)
    rois[:, 1:] = torch.tensor(boxes).to(device)
    
    crops = []
    for box in boxes:

        x1, y1, x2, y2 = map(int, box)

        crop = frame[y1:y2, x1:x2]

        if crop.size == 0:
            crop = np.zeros((256, 128, 3), dtype=np.uint8)

        crops.append(preprocess(crop))
    
    crops_tensor = torch.stack(crops).to(device)
    
    with torch.no_grad():
        embeddings = reid_model(crops_tensor)

    return embeddings

def update_with_detections_roi(self, detections: Detections, frame = None, roi_coef = 0.5) -> Detections:
    """
    Took this from supervision library
    """
    tensors = np.hstack(
        (
            detections.xyxy,
            detections.confidence[:, np.newaxis],
        )
    )
    tracks = self.update_with_tensors(tensors=tensors)

    if len(tracks) > 0:
        detection_bounding_boxes = np.asarray([det[:4] for det in tensors])
        track_bounding_boxes = np.asarray([track.tlbr for track in tracks])

        ious = box_iou_batch(detection_bounding_boxes, track_bounding_boxes)

        if frame is not None:
            
            detection_embeddings = extract_embeddings(frame, detection_bounding_boxes)
            track_embeddings = extract_embeddings(frame, track_bounding_boxes)

            cos_sim = torch.cosine_similarity(
                detection_embeddings.unsqueeze(1),  # (21, 1, D)
                track_embeddings.unsqueeze(0),      # (1, 20, D)
                dim=2
            ).cpu().numpy()

            ious = (1 - roi_coef) * ious + roi_coef * cos_sim

        iou_costs = 1 - ious

        matches, _, _ = matching.linear_assignment(iou_costs, 0.5)
        detections.tracker_id = np.full(len(detections), -1, dtype=int)
        for i_detection, i_track in matches:
            detections.tracker_id[i_detection] = int(
                tracks[i_track].external_track_id
            )

        return detections[detections.tracker_id != -1]

    else:
        detections = Detections.empty()
        detections.tracker_id = np.array([], dtype=int)

        return detections


def process_traching(source_dir, target_dir, use_roi = False, roi_coef = 0.5):

    image_paths = sorted([
        os.path.join(source_dir, f) for f in os.listdir(source_dir) 
        if f.lower().endswith(('.png', '.jpg', '.jpeg'))
    ])

    if not os.path.exists(target_dir):
        os.makedirs(target_dir)

    with open(mot_file_path, "w", encoding="utf-8") as mot_file:

        for frame_idx, img_path in enumerate(image_paths):
            frame = cv2.imread(img_path)
            if frame is None:
                continue

            results = model(frame, verbose=False, imgsz=1024)[0] #train image size
            detections = sv.Detections.from_ultralytics(results)

            if use_roi:
                detections = update_with_detections_roi(tracker, detections, frame, roi_coef)
            else:
                detections = update_with_detections_roi(tracker, detections)

            for i in range(len(detections.xyxy)):
                x1, y1, x2, y2 = detections.xyxy[i]

                if detections.tracker_id is not None:
                    track_id = detections.tracker_id[i]
                    conf = detections.confidence[i]
                    w = x2 - x1
                    h = y2 - y1

                    line = f"{frame_idx+1},{track_id},{x1:.2f},{y1:.2f},{w:.2f},{h:.2f},{conf:.4f},-1,-1,-1\n"
                    mot_file.write(line)

            if detections.tracker_id is not None:
                labels = [f"#{tid}" for tid in detections.tracker_id]
                annotated_frame = box_annotator.annotate(frame.copy(), detections=detections)
                annotated_frame = label_annotator.annotate(annotated_frame, detections=detections, labels=labels)
                
                output_path = os.path.join(target_dir, os.path.basename(img_path))
                cv2.imwrite(output_path, annotated_frame)

            if frame_idx % 50 == 0:
                print(f"Processed {frame_idx}/{len(image_paths)} images")

In [79]:
process_traching("VisDrone2019-MOT-test-dev/sequences/uav0000009_03358_v", "output_images_tracked", True, 0.3)

Processed 0/219 images
Processed 50/219 images
Processed 100/219 images
Processed 150/219 images
Processed 200/219 images


## c) MOT METRICKS

In [68]:

def evaluate_mot(gt_file, res_file):
    gt_cols = ['frame', 'id', 'x', 'y', 'w', 'h', 'conf', 'class', 'trunc', 'occ']
    gt_df = pd.read_csv(gt_file, header=None, names=gt_cols)

    res_cols = ['frame', 'id', 'x', 'y', 'w', 'h', 'conf', 'v1', 'v2', 'v3']
    res_df = pd.read_csv(res_file, header=None, names=res_cols)

    acc = mm.MOTAccumulator(auto_id=True)
    frames = sorted(gt_df['frame'].unique())
    
    for frame_id in frames:
        gt_frame = gt_df[gt_df['frame'] == frame_id]
        res_frame = res_df[res_df['frame'] == frame_id]

        gt_boxes = gt_frame[['x', 'y', 'w', 'h']].values
        res_boxes = res_frame[['x', 'y', 'w', 'h']].values if not res_frame.empty else np.array([])

        distances = mm.distances.iou_matrix(gt_boxes, res_boxes, max_iou=0.5)

        acc.update(
            gt_frame['id'].values,
            res_frame['id'].values if not res_frame.empty else [],
            distances
        )

    mh = mm.metrics.create()

    metrics_list = ['mota', 'motp', 'idf1', 'mostly_tracked', 'mostly_lost', 'num_switches']
    
    summary = mh.compute(acc, metrics=metrics_list, name='ByteTrack_Baseline')
    
    str_summary = mm.io.render_summary(
        summary, 
        formatters=mh.formatters, 
        namemap=mm.io.motchallenge_metric_names
    )
    print("\n--- MOT Evaluation Results ---")
    print(str_summary)
    return summary

In [80]:
evaluate_mot("VisDrone2019-MOT-test-dev/annotations/uav0000009_03358_v.txt", "detection_metrics.txt")


--- MOT Evaluation Results ---
                    MOTA  MOTP  IDF1 MT ML IDs
ByteTrack_Baseline 38.0% 0.265 43.6% 64 28 387


,mota,motp,idf1,mostly_tracked,mostly_lost,num_switches
ByteTrack_Baseline,0.380141,0.264909,0.435613,64,28,387


## Task 2

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import motmetrics as mm
import supervision as sv
from ultralytics import YOLO
from ultralyticsplus import YOLO

import torch
import torchvision.transforms as T
from torchvision.models import resnet50, ResNet50_Weights

from supervision.detection.core import Detections
from supervision.detection.utils.iou_and_nms import box_iou_batch
from supervision.tracker.byte_tracker.single_object_track import TrackState#, STrack, TrackState
from supervision.tracker.byte_tracker import matching
from supervision.tracker.byte_tracker.core import joint_tracks, sub_tracks, remove_duplicate_tracks

from ..src.strack import STrack

In [ ]:
class ROIByteTrack:
    def __init__(self,
                 model,
                 reid_model,
                 device = "cuda",):

        self.device = device
        self.reid_model = reid_model
        self.model = model
        
        self.reid_model.to(device)
        self.model.to(device)

        self.preprocess = T.Compose([
            T.ToPILImage(),
            T.Resize((256, 128)),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])


        self.box_annotator = sv.BoxAnnotator()
        self.label_annotator = sv.LabelAnnotator()
        self.tracker = sv.ByteTrack(
            track_activation_threshold=0.25,
            lost_track_buffer=30,
            minimum_matching_threshold=0.8,
            frame_rate=30
        )
    
    @staticmethod
    def embedding_distance(strack_pool, detections):
        
        print(strack_pool)
        print(detections)
        
        raise ValueError
        cos_sim = torch.cosine_similarity(
            detection_embeddings.unsqueeze(1),  # (21, 1, D)
            track_embeddings.unsqueeze(0),      # (1, 20, D)
            dim=2
        ).cpu().numpy()
        
    @staticmethod
    def update_with_tensors(self, tensors: np.ndarray, embeddings: np.ndarray = None, roi_coef = 0.5) -> list[STrack]:

        self.frame_id += 1
        activated_starcks = []
        refind_stracks = []
        lost_stracks = []
        removed_stracks = []

        scores = tensors[:, 4]
        bboxes = tensors[:, :4]

        remain_inds = scores > self.track_activation_threshold
        inds_low = scores > 0.1
        inds_high = scores < self.track_activation_threshold

        inds_second = np.logical_and(inds_low, inds_high)
        dets_second = bboxes[inds_second]
        dets = bboxes[remain_inds]
        scores_keep = scores[remain_inds]
        scores_second = scores[inds_second]

        if len(dets) > 0:
            """Detections"""
            detections = [
                STrack(
                    STrack.tlbr_to_tlwh(tlbr),
                    score_keep,
                    self.minimum_consecutive_frames,
                    self.shared_kalman,
                    self.internal_id_counter,
                    self.external_id_counter,
                    curr_feat=embeddings[remain_inds][i] if embeddings is not None else None #my change is here
                )
                for i, (tlbr, score_keep) in enumerate(zip(dets, scores_keep))
            ]
        else:
            detections = []

        """ Add newly detected tracklets to tracked_stracks"""
        unconfirmed = []
        tracked_stracks = []  # type: list[STrack]

        for track in self.tracked_tracks:
            if not track.is_activated:
                unconfirmed.append(track)
            else:
                tracked_stracks.append(track)

        """ Step 2: First association, with high score detection boxes"""
        strack_pool = joint_tracks(tracked_stracks, self.lost_tracks)
        # Predict the current location with KF
        STrack.multi_predict(strack_pool, self.shared_kalman)
        iou_dists = matching.iou_distance(strack_pool, detections)

        if embeddings is not None:
            # Тобі знадобиться функція для розрахунку 1 - cos_similarity
            emb_dists = ROIByteTrack.embedding_distance(strack_pool, detections) 
  
            dists = (1 - roi_coef) * iou_dists + roi_coef * emb_dists
        else:
            dists = iou_dists

        dists = matching.fuse_score(dists, detections)
        matches, u_track, u_detection = matching.linear_assignment(
            dists, thresh=self.minimum_matching_threshold
        )

        for itracked, idet in matches:
            track = strack_pool[itracked]
            det = detections[idet]
            if track.state == TrackState.Tracked:
                track.update(detections[idet], self.frame_id)
                activated_starcks.append(track)
            else:
                track.re_activate(det, self.frame_id)
                refind_stracks.append(track)

        """ Step 3: Second association, with low score detection boxes"""
        # association the untrack to the low score detections
        if len(dets_second) > 0:
            """Detections"""
            detections_second = [
                STrack(
                    STrack.tlbr_to_tlwh(tlbr),
                    score_second,
                    self.minimum_consecutive_frames,
                    self.shared_kalman,
                    self.internal_id_counter,
                    self.external_id_counter,
                )
                for (tlbr, score_second) in zip(dets_second, scores_second)
            ]
        else:
            detections_second = []
        r_tracked_stracks = [
            strack_pool[i]
            for i in u_track
            if strack_pool[i].state == TrackState.Tracked
        ]
        dists = matching.iou_distance(r_tracked_stracks, detections_second)
        matches, u_track, u_detection_second = matching.linear_assignment(
            dists, thresh=0.5
        )
        for itracked, idet in matches:
            track = r_tracked_stracks[itracked]
            det = detections_second[idet]
            if track.state == TrackState.Tracked:
                track.update(det, self.frame_id)
                activated_starcks.append(track)
            else:
                track.re_activate(det, self.frame_id)
                refind_stracks.append(track)

        for it in u_track:
            track = r_tracked_stracks[it]
            if not track.state == TrackState.Lost:
                track.state = TrackState.Lost
                lost_stracks.append(track)

        """Deal with unconfirmed tracks, usually tracks with only one beginning frame"""
        detections = [detections[i] for i in u_detection]
        dists = matching.iou_distance(unconfirmed, detections)

        dists = matching.fuse_score(dists, detections)
        matches, u_unconfirmed, u_detection = matching.linear_assignment(
            dists, thresh=0.7
        )
        for itracked, idet in matches:
            unconfirmed[itracked].update(detections[idet], self.frame_id)
            activated_starcks.append(unconfirmed[itracked])
        for it in u_unconfirmed:
            track = unconfirmed[it]
            track.state = TrackState.Removed
            removed_stracks.append(track)

        """ Step 4: Init new stracks"""
        for inew in u_detection:
            track = detections[inew]
            if track.score < self.det_thresh:
                continue
            track.activate(self.kalman_filter, self.frame_id)
            activated_starcks.append(track)
        """ Step 5: Update state"""
        for track in self.lost_tracks:
            if self.frame_id - track.frame_id > self.max_time_lost:
                track.state = TrackState.Removed
                removed_stracks.append(track)

        self.tracked_tracks = [
            t for t in self.tracked_tracks if t.state == TrackState.Tracked
        ]
        self.tracked_tracks = joint_tracks(self.tracked_tracks, activated_starcks)
        self.tracked_tracks = joint_tracks(self.tracked_tracks, refind_stracks)
        self.lost_tracks = sub_tracks(self.lost_tracks, self.tracked_tracks)
        self.lost_tracks.extend(lost_stracks)
        self.lost_tracks = sub_tracks(self.lost_tracks, self.removed_tracks)
        self.removed_tracks = removed_stracks
        self.tracked_tracks, self.lost_tracks = remove_duplicate_tracks(
            self.tracked_tracks, self.lost_tracks
        )
        output_stracks = [track for track in self.tracked_tracks if track.is_activated]

        return output_stracks

    def extract_embeddings(self, frame, boxes):
        """
        frame: cv2 image
        boxes: numpy array (N, 4) в форматі xyxy
        """
        if len(boxes) == 0:
            return torch.empty((0, 2048)).to(self.device)

        rois = torch.zeros((len(boxes), 5)).to(self.device)
        rois[:, 1:] = torch.tensor(boxes).to(self.device)
        
        crops = []
        for box in boxes:

            x1, y1, x2, y2 = map(int, box)

            crop = frame[y1:y2, x1:x2]

            if crop.size == 0:
                crop = np.zeros((256, 128, 3), dtype=np.uint8)

            crops.append(self.preprocess(crop))
        
        crops_tensor = torch.stack(crops).to(self.device)
        
        with torch.no_grad():
            embeddings = self.reid_model(crops_tensor)

        return embeddings

    def update_with_detections_roi(self, detections: Detections, embeddings, roi_coef = 0.5) -> Detections:
        """
        Took this from supervision library
        """
        tensors = np.hstack(
            (
                detections.xyxy,
                detections.confidence[:, np.newaxis],
            )
        )
        tracks = ROIByteTrack.update_with_tensors(self.tracker,
                                                  tensors=tensors,
                                                  embeddings=embeddings,
                                                  roi_coef = roi_coef)

        if len(tracks) > 0:
            detection_bounding_boxes = np.asarray([det[:4] for det in tensors])
            track_bounding_boxes = np.asarray([track.tlbr for track in tracks])

            ious = box_iou_batch(detection_bounding_boxes, track_bounding_boxes)

            iou_costs = 1 - ious

            matches, _, _ = matching.linear_assignment(iou_costs, 0.5)
            detections.tracker_id = np.full(len(detections), -1, dtype=int)
            for i_detection, i_track in matches:
                detections.tracker_id[i_detection] = int(
                    tracks[i_track].external_track_id
                )

            return detections[detections.tracker_id != -1]

        else:
            detections = Detections.empty()
            detections.tracker_id = np.array([], dtype=int)

            return detections


    def process_traching(self, 
                         source_dir, 
                         target_dir, 
                         mot_file_path = "detection_metrics.txt", 
                         use_roi = False, 
                         roi_coef = 0.5):

        image_paths = sorted([
            os.path.join(source_dir, f) for f in os.listdir(source_dir) 
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])

        if not os.path.exists(target_dir):
            os.makedirs(target_dir)

        with open(mot_file_path, "w", encoding="utf-8") as mot_file:

            for frame_idx, img_path in enumerate(image_paths):
                frame = cv2.imread(img_path)
                if frame is None:
                    continue

                results = self.model(frame, verbose=False, imgsz=1024)[0] #train image size
                detections = sv.Detections.from_ultralytics(results)

                if use_roi:
                    
                    boxes = self.model(frame).xyxy
                    
                    embeddings = extract_embeddings(frame, boxes)
                    
                    detections = self.update_with_detections_roi(detections, embeddings, roi_coef)
                else:
                    detections = self.update_with_detections_roi(detections)

                for i in range(len(detections.xyxy)):
                    x1, y1, x2, y2 = detections.xyxy[i]

                    if detections.tracker_id is not None:
                        track_id = detections.tracker_id[i]
                        conf = detections.confidence[i]
                        w = x2 - x1
                        h = y2 - y1

                        line = f"{frame_idx+1},{track_id},{x1:.2f},{y1:.2f},{w:.2f},{h:.2f},{conf:.4f},-1,-1,-1\n"
                        mot_file.write(line)

                if detections.tracker_id is not None:
                    labels = [f"#{tid}" for tid in detections.tracker_id]
                    annotated_frame = self.box_annotator.annotate(frame.copy(), detections=detections)
                    annotated_frame = self.label_annotator.annotate(annotated_frame, detections=detections, labels=labels)
                    
                    output_path = os.path.join(target_dir, os.path.basename(img_path))
                    cv2.imwrite(output_path, annotated_frame)

                if frame_idx % 50 == 0:
                    print(f"Processed {frame_idx}/{len(image_paths)} images")
    
    @staticmethod
    def evaluate_mot(gt_file, res_file, verbose = True):
        gt_cols = ['frame', 'id', 'x', 'y', 'w', 'h', 'conf', 'class', 'trunc', 'occ']
        gt_df = pd.read_csv(gt_file, header=None, names=gt_cols)

        res_cols = ['frame', 'id', 'x', 'y', 'w', 'h', 'conf', 'v1', 'v2', 'v3']
        res_df = pd.read_csv(res_file, header=None, names=res_cols)

        acc = mm.MOTAccumulator(auto_id=True)
        frames = sorted(gt_df['frame'].unique())
        
        for frame_id in frames:
            gt_frame = gt_df[gt_df['frame'] == frame_id]
            res_frame = res_df[res_df['frame'] == frame_id]

            gt_boxes = gt_frame[['x', 'y', 'w', 'h']].values
            res_boxes = res_frame[['x', 'y', 'w', 'h']].values if not res_frame.empty else np.array([])

            distances = mm.distances.iou_matrix(gt_boxes, res_boxes, max_iou=0.5)

            acc.update(
                gt_frame['id'].values,
                res_frame['id'].values if not res_frame.empty else [],
                distances
            )

        mh = mm.metrics.create()

        metrics_list = ['mota', 'motp', 'idf1', 'mostly_tracked', 'mostly_lost', 'num_switches']
        
        summary = mh.compute(acc, metrics=metrics_list, name='ByteTrack_Baseline')
        
        if verbose:
            str_summary = mm.io.render_summary(
                summary, 
                formatters=mh.formatters, 
                namemap=mm.io.motchallenge_metric_names
            )
            print("\n--- MOT Evaluation Results ---")
            print(str_summary)

        return summary



In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = YOLO('mshamrai/yolov8n-visdrone') # model for detection
model.overrides['conf'] = 0.25
model.overrides['iou'] = 0.45
model.overrides['agnostic_nms'] = False
model.overrides['max_det'] = 1000

reid_model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1) # model for feature extraction
reid_model.fc = torch.nn.Identity()
reid_model.to(device).eval()

track = ROIByteTrack(model = model,
                        reid_model=reid_model,
                        device=device)

mot_metricks_path = "detection_metrics.txt" # path where mot metricks will be stored, maybe will rework this

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/mshamrai/yolov8n-visdrone/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/mshamrai/yolov8n-visdrone/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/mshamrai/yolov8n-visdrone/28ca5101e5e6090c6beba15eb79d038d0befad6d/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/mshamrai/yolov8n-visdrone/resolve/main/best.pt "HTTP/1.1 302 Found"


In [ ]:
track.process_traching("../task1/VisDrone2019-MOT-test-dev/sequences/uav0000009_03358_v", # path to dataset image sequence
                        "output_images_tracked",  #p ath to output tracked objects on the image
                        mot_metricks_path,
                        use_roi = True,
                        roi_coef = 0.3
                        )

In [ ]:
track.evaluate_mot("../task1/VisDrone2019-MOT-test-dev/annotations/uav0000009_03358_v.txt", #path to annotations
                    mot_metricks_path,
                    verbose = True
                    )